# TII UAV — Audio-Only Dataset Analysis + Mamba

A clean **audio-only** notebook. No IMU features, no video, and no multimodal comparisons.

Goals:
1. Inspect raw audio structure and quality.
2. Visualize waveform, channel/interleaving structure and spectra.
3. Identify class-dependent frequency regions.
4. Build one leakage-safe audio representation.
5. Train a standalone real-Mamba audio classifier and report its strength.

In [ ]:
import os, json, math, random, subprocess
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED)
BASE=Path("/content/tii_uav_dataset")
ROOT=BASE/"Dataset"
URL="https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"

if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1",URL,str(BASE)],check=True)

AUDIO_SR=48000.0
WINDOW_SEC=2.56
STEP_SEC=1.28
TA=64

print("Dataset:",ROOT)

In [ ]:
# Discover missions
rows=[]
for d in sorted(ROOT.iterdir()):
    if not d.is_dir() or not d.name[:1].isdigit(): continue
    label=int(d.name.split("-")[0])
    for m in sorted(d.iterdir()):
        if m.is_dir():
            rows.append({
                "class":label,"mission":m.name,"path":str(m),
                "audio":bool(list(m.glob("*AudioBuffer*.jsonl"))),
                "imu":bool(list(m.glob("*SensorCombined*.jsonl")))
            })
missions=pd.DataFrame(rows)

print("Missions:",len(missions))
display(missions.groupby("class").size().rename("missions").to_frame())
display(pd.crosstab(missions["class"],[missions["audio"],missions["imu"]]))

In [ ]:
def read_audio(mission):
    hits=list(Path(mission).glob("*AudioBuffer*.jsonl"))
    if not hits: return None
    chunks=[]
    with open(hits[0],encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try: r=json.loads(line)
            except: continue
            x=r.get("audio_data",r.get("buffer",r.get("data",[])))
            if len(x): chunks.append(np.asarray(x,dtype=np.float32))
    if not chunks: return None
    lens=[len(x) for x in chunks]
    common=Counter(lens).most_common(1)[0][0]
    chunks=[x for x in chunks if len(x)==common]
    return np.stack(chunks)

# Raw integrity scan
stats=[]
for _,r in missions[missions.audio].iterrows():
    x=read_audio(r.path)
    if x is None: continue
    z=x.ravel()
    stats.append({
        "class":r["class"],"mission":r["mission"],"chunks":len(x),
        "chunk_len":x.shape[1],"mean":z.mean(),"std":z.std(),
        "rms":np.sqrt(np.mean(z*z)),"zero_frac":np.mean(z==0),
        "min":z.min(),"max":z.max()
    })
raw=pd.DataFrame(stats)
display(raw.groupby("class").agg({"chunks":"sum","chunk_len":"median",
                                  "std":"mean","rms":"mean","zero_frac":"mean"}))
display(raw[["min","max","mean","std","rms","zero_frac"]].describe())

In [ ]:
# Representative raw waveforms
fig,axs=plt.subplots(5,1,figsize=(12,10))
for cls,ax in enumerate(axs):
    r=raw[raw["class"]==cls].iloc[0]
    x=read_audio(missions[missions.mission==r.mission].iloc[0].path)[0]
    ax.plot(x[:1200],lw=.7)
    ax.set_title(f"Class {cls} — mission {r.mission}")
    ax.set_ylabel("amplitude")
axs[-1].set_xlabel("raw sample index")
plt.tight_layout(); plt.show()

# Interleaving diagnostic: six candidate slots
profiles=[]
for _,r in missions[missions.audio].iterrows():
    x=read_audio(r.path)
    if x is None: continue
    z=x[0]
    n=6
    z=z[:len(z)//n*n].reshape(-1,n)
    for k in range(n):
        profiles.append([r["class"],r["mission"],k,z[:,k].std(),
                         np.sqrt(np.mean(z[:,k]**2)),np.mean(z[:,k]==0)])
prof=pd.DataFrame(profiles,columns=["class","mission","slot","std","rms","zero"])
display(prof.groupby("slot")[["std","rms","zero"]].mean())

In [ ]:
def fft_power(x):
    spec=np.fft.rfft(x,axis=1)
    return np.fft.rfftfreq(x.shape[1],1/AUDIO_SR),np.abs(spec)**2

# Class-conditional normalized spectra
class_spec={}
for cls in sorted(missions["class"].unique()):
    acc=[]
    for _,r in missions[(missions["class"]==cls)&missions.audio].iterrows():
        x=read_audio(r.path)
        if x is None: continue
        x=x[:min(100,len(x))]
        f,p=fft_power(x)
        p/=p.sum(axis=1,keepdims=True)+1e-12
        acc.append(p.mean(0))
    class_spec[cls]=np.mean(acc,0)

plt.figure(figsize=(13,5))
for cls,s in class_spec.items(): plt.plot(f,s,label=f"class {cls}")
plt.xlabel("Hz"); plt.ylabel("normalized power"); plt.title("Mean spectrum by fault class")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()

for lo,hi in [(0,2000),(10000,16000),(25000,32000),(38000,46000)]:
    plt.figure(figsize=(12,4))
    for cls,s in class_spec.items():
        q=(f>=lo)&(f<=hi); plt.plot(f[q],s[q],label=f"class {cls}")
    plt.xlim(lo,hi); plt.xlabel("Hz"); plt.ylabel("normalized power")
    plt.title(f"Spectrum: {lo/1000:.1f}–{hi/1000:.1f} kHz")
    plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()

In [ ]:
# Mission-level spectral discriminativeness
X=[]; y=[]
for _,r in missions[missions.audio].iterrows():
    x=read_audio(r.path)
    if x is None: continue
    f,p=fft_power(x[:min(100,len(x))])
    p/=p.sum(axis=1,keepdims=True)+1e-12
    X.append(p.mean(0)); y.append(r["class"])
X=np.stack(X); y=np.array(y)

grand=X.mean(0); between=np.zeros(X.shape[1]); within=np.zeros(X.shape[1])
for c in np.unique(y):
    xc=X[y==c]; between+=len(xc)*(xc.mean(0)-grand)**2
    within+=((xc-xc.mean(0))**2).sum(0)
score=between/(within+1e-12)

idx=np.argsort(score)[::-1][:100]
display(pd.DataFrame({"bin":idx,"Hz":f[idx],"score":score[idx]}).head(30))

plt.figure(figsize=(13,4))
plt.plot(f,score); plt.yscale("log")
plt.xlabel("Hz"); plt.ylabel("between/within class variance")
plt.title("Exploratory frequency discriminativeness")
plt.grid(alpha=.2); plt.tight_layout(); plt.show()

In [ ]:
# Temporal spectral stability
out=[]
for _,r in missions[missions.audio].iterrows():
    x=read_audio(r.path)
    if x is None or len(x)<3: continue
    x=x[:min(500,len(x))]
    f,p=fft_power(x); p/=p.sum(axis=1,keepdims=True)+1e-12
    cs=np.sum(p[:-1]*p[1:],1)/(np.linalg.norm(p[:-1],axis=1)*np.linalg.norm(p[1:],axis=1)+1e-12)
    centroid=(p*f[None,:]).sum(1)
    ent=-(p*np.log(p+1e-12)).sum(1)
    out.append([r["class"],r["mission"],cs.mean(),cs.std(),centroid.mean(),
                centroid.std(),ent.mean(),np.sqrt(np.mean(x*x)),np.std(np.sqrt(np.mean(x*x,1)))])
td=pd.DataFrame(out,columns=["class","mission","neighbor_cos","cos_std",
                             "centroid_hz","centroid_std","entropy","rms","rms_std"])
display(td.groupby("class").mean(numeric_only=True))
for col in ["neighbor_cos","centroid_hz","entropy","rms"]:
    plt.figure(figsize=(9,4)); td.boxplot(column=col,by="class")
    plt.title(col); plt.suptitle(""); plt.tight_layout(); plt.show()

## Final audio representation

For the standalone Mamba run we use the representation that follows the earlier frequency-bin finding, but selection is made **from training missions only**.

- rFFT magnitude
- square-root scaling
- 64 strongest training-spectrum frequency bins
- 2.56 s windows, 1.28 s stride
- 64 temporal audio tokens
- train-only normalization

The exploratory plots above are for understanding the dataset; they are not used to tune against the test set.

In [ ]:
# Mission-level split
rng=np.random.default_rng(SEED)
assign=[]
for c in sorted(missions["class"].unique()):
    ids=missions[missions["class"]==c].mission.tolist()
    rng.shuffle(ids); n=len(ids)
    nt=round(.70*n); nv=round(.15*n)
    for i,m in enumerate(ids):
        assign.append([m,"train" if i<nt else ("val" if i<nt+nv else "test")])
split=pd.DataFrame(assign,columns=["mission","split"])
missions=missions.merge(split,on="mission",how="left")
display(pd.crosstab(missions["class"],missions["split"]))

# Training-only spectral selection
acc=[]; f=None
for _,r in missions[(missions.split=="train")&missions.audio].iterrows():
    x=read_audio(r.path)
    if x is None: continue
    f,p=fft_power(x); p/=p.sum(axis=1,keepdims=True)+1e-12
    acc.append(p.mean(0))
train_spec=np.mean(acc,0)
bins=np.sort(np.argsort(train_spec)[::-1][:64])
freqs=f[bins]
print("Selected frequencies (Hz):")
print(np.round(freqs,1))
plt.figure(figsize=(12,3)); plt.stem(freqs,np.ones(len(freqs)))
plt.xlabel("Hz"); plt.title("64 frequency bins selected using TRAIN missions only")
plt.tight_layout(); plt.show()

In [ ]:
# Build audio windows
def sensor_duration(mission):
    hits=list(Path(mission).glob("*SensorCombined*.jsonl"))
    if not hits: return None
    ts=[]
    with open(hits[0],encoding="utf-8") as f:
        for line in f:
            try: ts.append(float(json.loads(line)["timestamp"])/1e6)
            except: pass
    return max(ts)-min(ts) if len(ts)>1 else None

def interp_seq(x,n=TA):
    if len(x)==n: return x
    a=np.linspace(0,1,len(x)); b=np.linspace(0,1,n)
    return np.stack([np.interp(b,a,x[:,j]) for j in range(x.shape[1])],1)

windows=[]
for _,r in missions[missions.audio].iterrows():
    x=read_audio(r.path)
    dur=sensor_duration(r.path)
    if x is None or dur is None or dur<=0: continue
    dt=dur/len(x); times=np.arange(len(x))*dt
    spec=np.sqrt(np.abs(np.fft.rfft(x,axis=1))[:,bins]).astype(np.float32)

    t0=0.0
    while t0+WINDOW_SEC<=times[-1]:
        mask=(times>=t0)&(times<=t0+WINDOW_SEC)
        if mask.sum()>=4:
            windows.append({"x":interp_seq(spec[mask]),"y":int(r["class"]),
                            "mission":r.mission,"split":r.split})
        t0+=STEP_SEC

train=np.stack([w["x"] for w in windows if w["split"]=="train"])
mu=train.mean((0,1),keepdims=True); sd=train.std((0,1),keepdims=True)+1e-6
for w in windows: w["x"]=((w["x"]-mu)/sd).astype(np.float32)

print("Windows:",len(windows))
print(Counter(w["split"] for w in windows))
print("Input shape:",windows[0]["x"].shape)

In [ ]:
# Real Mamba
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,roc_auc_score,classification_report,confusion_matrix

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

try:
    from mamba_ssm import Mamba
except ImportError as e:
    raise RuntimeError("Install real mamba-ssm on the GPU runtime, then restart.") from e

class DS(Dataset):
    def __init__(self,w): self.w=w
    def __len__(self): return len(self.w)
    def __getitem__(self,i): return torch.tensor(self.w[i]["x"]),torch.tensor(self.w[i]["y"])

tr=DataLoader(DS([w for w in windows if w["split"]=="train"]),32,shuffle=True)
va=DataLoader(DS([w for w in windows if w["split"]=="val"]),64)
te=DataLoader(DS([w for w in windows if w["split"]=="test"]),64)

class AudioMamba(nn.Module):
    def __init__(self):
        super().__init__()
        d=256
        self.proj=nn.Sequential(nn.Linear(64,d),nn.LayerNorm(d),nn.GELU(),nn.Linear(d,d))
        self.blocks=nn.ModuleList([nn.ModuleDict({
            "norm":nn.LayerNorm(d),
            "mamba":Mamba(d_model=d,d_state=16,d_conv=4,expand=2)
        }) for _ in range(4)])
        self.pool=nn.Linear(d,1); self.cls=nn.Linear(d,5)
    def forward(self,x):
        x=self.proj(x)
        for b in self.blocks: x=x+b["mamba"](b["norm"](x))
        h=x
        a=torch.softmax(self.pool(h).squeeze(-1),1)
        return self.cls((h*a.unsqueeze(-1)).sum(1))

model=AudioMamba().to(device)
opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)
loss_fn=nn.CrossEntropyLoss()
print("Parameters:",sum(p.numel() for p in model.parameters()))

In [ ]:
def epoch(loader,train_mode):
    model.train(train_mode); ys=[]; ps=[]; pp=[]; total=0
    for x,y in loader:
        x=x.to(device); y=y.to(device)
        if train_mode: opt.zero_grad(set_to_none=True)
        z=model(x); loss=loss_fn(z,y)
        if train_mode:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        total+=loss.item()*len(y)
        ys+=y.cpu().tolist(); ps+=z.argmax(1).cpu().tolist(); pp.append(torch.softmax(z,1).detach().cpu().numpy())
    return {"loss":total/len(loader.dataset),"f1":f1_score(ys,ps,average="macro"),
            "acc":accuracy_score(ys,ps),"bal":balanced_accuracy_score(ys,ps),
            "y":np.array(ys),"p":np.array(ps),"prob":np.concatenate(pp)}

best=-1; state=None; bad=0; hist=[]
for ep in range(1,41):
    a=epoch(tr,True); b=epoch(va,False)
    hist.append([ep,a["f1"],b["f1"]])
    print(f"{ep:02d} trainF1={a['f1']:.4f} valF1={b['f1']:.4f}")
    if b["f1"]>best:
        best=b["f1"]; state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
    else: bad+=1
    if bad>=8: break

model.load_state_dict(state)
res=epoch(te,False)
auc=roc_auc_score(res["y"],res["prob"],multi_class="ovr",average="macro")
print("\nAUDIO-ONLY MAMBA")
print("Accuracy:",res["acc"])
print("Macro-F1:",res["f1"])
print("Balanced Accuracy:",res["bal"])
print("Macro-AUROC:",auc)
print(classification_report(res["y"],res["p"],digits=4))

In [ ]:
cm=confusion_matrix(res["y"],res["p"])
plt.figure(figsize=(6,5)); plt.imshow(cm)
plt.colorbar(); plt.xticks(range(5)); plt.yticks(range(5))
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Audio-only Mamba confusion matrix")
for i in range(5):
    for j in range(5): plt.text(j,i,cm[i,j],ha="center",va="center")
plt.tight_layout(); plt.show()

h=pd.DataFrame(hist,columns=["epoch","train_f1","val_f1"])
plt.figure(figsize=(8,4)); plt.plot(h.epoch,h.train_f1,label="train"); plt.plot(h.epoch,h.val_f1,label="validation")
plt.xlabel("Epoch"); plt.ylabel("Macro-F1"); plt.title("Audio-only Mamba")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()

## What this notebook should give us

**Dataset characterization**
- raw audio integrity and interleaving structure
- amplitude/RMS behavior
- class-conditional spectra
- discriminative frequency regions
- temporal spectral stability
- 64 selected frequencies

**Standalone strength**
- Audio-only Mamba Accuracy
- Macro-F1
- Balanced Accuracy
- Macro-AUROC
- per-class F1/recall
- confusion matrix
- parameter count

Only after this is stable should we compare audio with IMU or fusion, or run the controlled temporal-shift experiment.